In [ ]:
!pip install gliner

In [ ]:
!pip install gliner -q

import zipfile
from pathlib import Path

with zipfile.ZipFile("data_ai.zip", "r") as zip_ref:
    zip_ref.extractall("data_ai_extracted")

with zipfile.ZipFile("test_data.zip", "r") as zip_ref:
    zip_ref.extractall("test_data_extracted")


In [ ]:
def to_gliner_format(dataset):
    return [{"tokenized_text": item["tokens"], "ner": item["ner"]} for item in dataset]

train_dataset = to_gliner_format(train_dataset)
eval_dataset = to_gliner_format(eval_dataset)

print(train_dataset[0].keys())

In [ ]:
import re
import random
from pathlib import Path

TAG_RE = re.compile(r"<per>(.*?)</?per>", re.DOTALL | re.IGNORECASE)

def parse_tagged_text(raw_text):
    clean_chars = []
    entities_char_spans = []
    pos = 0
    clean_pos = 0

    for match in re.finditer(r"(<per>)(.*?)(</?per>)", raw_text, re.IGNORECASE | re.DOTALL):
        prefix = raw_text[pos:match.start()]
        clean_chars.append(prefix)
        clean_pos += len(prefix)

        name_text = match.group(2)
        start_char = clean_pos
        clean_chars.append(name_text)
        clean_pos += len(name_text)
        end_char = clean_pos

        if name_text.strip():
            entities_char_spans.append((start_char, end_char))

        pos = match.end()

    clean_chars.append(raw_text[pos:])
    clean_doc = "".join(clean_chars)

    sentences = re.split(r"(?<=\.)\s+", clean_doc)
    parsed_samples = []
    current_char_offset = 0

    for sent in sentences:
        if not sent.strip():
            continue

        sent_start = clean_doc.find(sent, current_char_offset)
        sent_end = sent_start + len(sent)
        current_char_offset = sent_end

        words = []
        word_spans = []
        for m in re.finditer(r"\S+", sent):
            words.append(m.group(0))
            word_spans.append((sent_start + m.start(), sent_start + m.end()))

        if not words:
            continue

        ner_spans = []
        for ent_start, ent_end in entities_char_spans:
            if ent_start >= sent_start and ent_end <= sent_end:
                tok_start = None
                tok_end = None
                for idx, (w_s, w_e) in enumerate(word_spans):
                    if w_s <= ent_start < w_e:
                        tok_start = idx
                    if w_s < ent_end <= w_e:
                        tok_end = idx
                if tok_start is not None and tok_end is not None:
                    ner_spans.append([tok_start, tok_end, "person"])

        parsed_samples.append({
            "tokens": words,
            "ner": ner_spans
        })

    return parsed_samples

train_files = list(Path("data_ai_extracted").rglob("*.txt"))
gliner_dataset = []

for file_path in train_files:
    content = file_path.read_text(encoding="utf-8", errors="replace")
    gliner_dataset.extend(parse_tagged_text(content))

random.seed(42)
random.shuffle(gliner_dataset)
split_idx = int(0.8 * len(gliner_dataset))

train_dataset = gliner_dataset[:split_idx]
eval_dataset = gliner_dataset[split_idx:]

print(f"Loaded {len(train_files)} files.")
print(f"Dataset ready -> Train samples: {len(train_dataset)} | Eval samples: {len(eval_dataset)}")

In [ ]:
import gc
import torch
from gliner import GLiNER
from gliner.training import Trainer as GLiNERTrainer
from gliner.training import TrainingArguments as GLiNERTrainingArguments

if "model" in locals():
    del model
if "trainer" in locals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "medieval-data/gliner_multi-v2.1-medieval-latin"
model = GLiNER.from_pretrained(MODEL_ID)

data_collator = model.data_collator_class(
    model.config,
    data_processor=model.data_processor,
    prepare_labels=True,
)

training_args = GLiNERTrainingArguments(
    output_dir="./gliner-medieval-latin-person",
    learning_rate=3e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

trainer = GLiNERTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
for name, ds in [("train", train_dataset), ("eval", eval_dataset)]:
    print(name, type(ds), len(ds))
    bad = [i for i, ex in enumerate(ds) if "tokenized_text" not in ex or "ner" not in ex]
    print("  rows missing keys:", len(bad), bad[:10])
    print("  first row keys:", list(ds[0].keys()))

batch = [train_dataset[i] for i in range(4)]
out = data_collator(batch)
print({k: getattr(v, "shape", type(v)) for k, v in out.items()})

In [ ]:
import time
import re
import torch
from pathlib import Path
from gliner import GLiNER

device = "cuda:0" if torch.cuda.is_available() else "cpu"

BASE_MODEL_ID = "urchade/gliner_multi-v2.1"
base_model = GLiNER.from_pretrained(BASE_MODEL_ID).to(device)
base_model.eval()

model.to(device)
model.eval()

def evaluate_gliner_model(model_instance, corpus, labels=["person"], threshold=0.4):
    t0 = time.perf_counter()
    all_preds = []
    true_positives = 0
    total_gold = sum(len(doc["gold"]) for doc in corpus)
    total_words = sum(len(doc["text"].split()) for doc in corpus)

    for doc in corpus:
        text = doc["text"]
        words = text.split()
        chunk_size = 250
        doc_preds = []

        for i in range(0, len(words), chunk_size):
            chunk = " ".join(words[i:i + chunk_size])
            entities = model_instance.predict_entities(chunk, labels, threshold=threshold)
            for ent in entities:
                doc_preds.append(ent["text"].strip())

        all_preds.extend(doc_preds)

        matched_indices = set()
        for gold in doc["gold"]:
            gold_compact = re.sub(r"[^\w]", "", gold.lower())
            for idx, pred in enumerate(doc_preds):
                if idx in matched_indices:
                    continue
                pred_compact = re.sub(r"[^\w]", "", pred.lower())
                if gold_compact in pred_compact or pred_compact in gold_compact:
                    true_positives += 1
                    matched_indices.add(idx)
                    break

    elapsed = time.perf_counter() - t0
    pred_count = len(all_preds)
    precision = true_positives / pred_count if pred_count > 0 else 0.0
    recall = true_positives / total_gold if total_gold > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    wps = total_words / elapsed if elapsed > 0 else 0.0

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "tp": true_positives,
        "total_gold": total_gold,
        "pred_count": pred_count,
        "wps": wps
    }

print("Benchmarking Base GLiNER...")
base_metrics = evaluate_gliner_model(base_model, test_corpus)

print("Benchmarking Fine-Tuned GLiNER...")
ft_metrics = evaluate_gliner_model(model, test_corpus)

print("\n" + "=" * 70)
print(f"{'Metric':<25} | {'Base GLiNER':<18} | {'Fine-Tuned GLiNER':<18}")
print("-" * 70)
print(f"{'F1 Score':<25} | {base_metrics['f1']:<18.4f} | {ft_metrics['f1']:<18.4f}")
print(f"{'Precision':<25} | {base_metrics['precision']:<18.4f} | {ft_metrics['precision']:<18.4f}")
print(f"{'Recall':<25} | {base_metrics['recall']:<18.4f} | {ft_metrics['recall']:<18.4f}")
print(f"{'True Positives':<25} | {base_metrics['tp']}/{base_metrics['total_gold']:<16} | {ft_metrics['tp']}/{ft_metrics['total_gold']:<16}")
print(f"{'Predicted Spans':<25} | {base_metrics['pred_count']:<18} | {ft_metrics['pred_count']:<18}")
print(f"{'Throughput (wps)':<25} | {base_metrics['wps']:<18.1f} | {ft_metrics['wps']:<18.1f}")
print("=" * 70)

In [ ]:
!pip install huggingface_hub -q

from huggingface_hub import login

login()

In [ ]:
REPO_NAME = "gliner_lviv"

save_directory = "./gliner_ner_lviv"
model.save_pretrained(save_directory)

model.push_to_hub(
    REPO_NAME,
    private=False
)

print(f"Model successfully uploaded to Hugging Face Hub under repository: {REPO_NAME}")